<a href="https://colab.research.google.com/github/Ewerton352/dio-lab-open-source/blob/Ewerton352/Criando_um_Sistema_de_Reconhecimento_Facial_do_Zero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Detecção de objetos YOLOv4 na webcam no Google Colab
Este notebook percorrerá todas as etapas para realizar detecções de objetos YOLOv4 em sua webcam enquanto estiver no Google Colab. **negrito**

In [ ]:
# import necessários
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import html
import time
import matplotlib.pyplot as plt
%matplotlib inline

Clonagem e configuração da Darknet para YOLOv4
Usaremos o famoso repositório darknet do AlexeyAB para realizar detecções YOLOv4. **negrito**

In [ ]:
# clone darknet repositorio
!git clone https://github.com/AlexeyAB/darknet

In [ ]:
# # altere o makefile para ter GPU, OPENCV e LIBSO habilitados
%cd darknet
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
!sed -i 's/LIBSO=0/LIBSO=1/' Makefile

In [ ]:
# make darknet (constrói darknet para que você possa usar o arquivo darknet.py e ter suas dependências)
!make

In [ ]:
# Obtenha o arquivo de pesos YOLOV4 dimensionado que é pré-treinado para detectar 80 classes (objetos) do Google Drive compartilhado
!wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id=1V3vsIaxAlGWvK4Aar9bAiK5U0QFttKwq' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id=1V3vsIaxAlGWvK4Aar9bAiK5U0QFttKwq" -O yolov4-csp.weights && rm -rf /tmp/cookies.txt

**Darknet para Python**
**Para utilizar o YOLOv4 com código Python, usaremos algumas das funções** **pré-construídas encontradas no darknet.py importando as funções para nossa** **estação de trabalho. Sinta-se à vontade para verificar o arquivo darknet.py** **para ver as definições de função em detalhes!**

In [ ]:
# importar funções darknet para realizar detecções de objetos
from darknet import *
# carga em nossa rede de arquitetura YOLOv4
network, class_names, class_colors = load_network("cfg/yolov4-csp.cfg", "cfg/coco.data", "yolov4-csp.weights")
width = network_width(network)
height = network_height(network)

# função auxiliar darknet para executar a detecção na imagem
def darknet_helper(img, width, height):
  darknet_image = make_image(width, height, 3)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  img_resized = cv2.resize(img_rgb, (width, height),
                              interpolation=cv2.INTER_LINEAR)

# obter proporções de imagem para converter caixas delimitadoras para o tamanho adequado
  img_height, img_width, _ = img.shape
  width_ratio = img_width/width
  height_ratio = img_height/height

# Execute o modelo na imagem de estilo darknet para obter detecções
  copy_image_from_bytes(darknet_image, img_resized.tobytes())
  detections = detect_image(network, class_names, darknet_image)
  free_image(darknet_image)
  return detections, width_ratio, height_ratio


**Exemplo de YOLOv4 na imagem de teste**
**Vamos garantir que nosso modelo tenha sido carregado com êxito e que possamos** **fazer detecções corretamente em uma imagem de teste.**

In [ ]:
# execute o teste em person.jpg imagem que vem com o repositório
image = cv2.imread("data/person.jpg")
detections, width_ratio, height_ratio = darknet_helper(image, width, height)

for label, confidence, bbox in detections:
  left, top, right, bottom = bbox2points(bbox)
  left, top, right, bottom = int(left * width_ratio), int(top * height_ratio), int(right * width_ratio), int(bottom * height_ratio)
  cv2.rectangle(image, (left, top), (right, bottom), class_colors[label], 2)
  cv2.putText(image, "{} [{:.2f}]".format(label, float(confidence)),
                    (left, top - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    class_colors[label], 2)
cv2_imshow(image)

**Funções auxiliares**
**Aqui estão algumas funções auxiliares definidas que serão usadas para** **converter facilmente entre diferentes tipos de imagem em nossas etapas** **posteriores.**

In [ ]:
# função para converter o objeto JavaScript em uma imagem OpenCV
def js_to_image(js_reply):
  """
  Params:
          js_reply: JavaScript object containing image from webcam
  Returns:
          img: OpenCV BGR image
  """
 # função para converter o objeto JavaScript em uma imagem OpenCV
  image_bytes = b64decode(js_reply.split(',')[1])
  # converter bytes em array numpy
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
 # decodificar o array numpy na imagem BGR do OpenCV
  img = cv2.imdecode(jpg_as_np, flags=1)

  return img

# função para converter a imagem da caixa delimitadora do retângulo OpenCV em string de 64 bytes de base a ser sobreposta no fluxo de vídeo
def bbox_to_bytes(bbox_array):
  """
  Params:
          bbox_array: Numpy array (pixels) containing rectangle to overlay on video stream.
  Returns:
        bytes: Base64 image byte string
  """
# converter array em imagem PIL
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  iobuf = io.BytesIO()
# formatar bbox em png para retorno
  bbox_PIL.save(iobuf, format='png')
#formato de retorno de string
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))

  return bbox_bytes

YOLOv4 em Imagens Wecam
Executar o YOLOv4 em imagens tiradas da webcam é bastante simples. Utilizaremos o código nos trechos de código do Google Colab que possui uma variedade de funções de código úteis para executar várias tarefas.

Usaremos o trecho de código para o Camera Capture, que executa o código JavaScript para utilizar a webcam do seu computador. O trecho de código tirará uma foto da webcam, que passaremos para nosso modelo YOLOv4 para detecção de objetos.

Abaixo está uma função para tirar a foto da webcam usando JavaScript e executar o YOLOv4 nela

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Redimensione a saída para caber no elemento de vídeo.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Aguarde até que o Capture seja clicado.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)

   # obter dados da foto
  data = eval_js('takePhoto({})'.format(quality))
  # obter imagem no formato OpenCV
  img = js_to_image(data)

  # chame nosso ajudante da darknet na imagem da webcam
  detections, width_ratio, height_ratio = darknet_helper(img, width, height)

  # percorra as detecções e desenhe-as na imagem da webcam
  for label, confidence, bbox in detections:
    left, top, right, bottom = bbox2points(bbox)
    left, top, right, bottom = int(left * width_ratio), int(top * height_ratio), int(right * width_ratio), int(bottom * height_ratio)
    cv2.rectangle(img, (left, top), (right, bottom), class_colors[label], 2)
    cv2.putText(img, "{} [{:.2f}]".format(label, float(confidence)),
                      (left, top - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                      class_colors[label], 2)
  # salve a imagem
  cv2.imwrite(filename, img)

  return filename

In [ ]:
try:
  filename = take_photo('photo.jpg')
  print('Saved to {}'.format(filename))

  # Mostre a imagem que acabou de ser tirada.
  display(Image(filename))
except Exception as err:
 # Erros serão lançados se o usuário não tiver uma webcam ou se não tiver
# conceda permissão à página para acessá-la.
  print(str(err))